# Imports, paths

In [50]:
from pathlib import Path
from urllib.request import urlretrieve
import re
import os
import ast
import pandas as pd
import numpy as np
from IPython.display import display
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# If the notebook is in multilingual/notebooks, cwd is usually:
# <repo>/multilingual/notebooks
NOTEBOOK_DIR = Path.cwd()

# Go two levels up:
# <repo>/multilingual/notebooks -> <repo>
REPO_ROOT = NOTEBOOK_DIR.parents[1]
MULTILINGUAL_DIR = REPO_ROOT / "multilingual"
RAW_ROOT = MULTILINGUAL_DIR / "data" / "raw" / "cultsportqa"
RAW_CSV_ROOT = MULTILINGUAL_DIR / "data" / "raw" / "cultsportqa_csv"
PROCESSED_ROOT = MULTILINGUAL_DIR / "data" / "processed" / "cultsportqa"
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/M-Groot7/CultSportQA/main"

COUNTRIES = [
    "China",
    "India",
]

GROUPS = ["HBQ", "RBQ", "SBQ"]

for p in [RAW_ROOT, RAW_CSV_ROOT, PROCESSED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:    ", NOTEBOOK_DIR)
print("REPO_ROOT:       ", REPO_ROOT)
print("MULTILINGUAL_DIR:", MULTILINGUAL_DIR)
print("RAW_ROOT:        ", RAW_ROOT)
print("RAW_CSV_ROOT:    ", RAW_CSV_ROOT)
print("PROCESSED_ROOT:  ", PROCESSED_ROOT)

NOTEBOOK_DIR:     /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/notebooks
REPO_ROOT:        /scratch/workspace/emma/standard-project-m2-mystery-machine
MULTILINGUAL_DIR: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual
RAW_ROOT:         /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cultsportqa
RAW_CSV_ROOT:     /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cultsportqa_csv
PROCESSED_ROOT:   /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cultsportqa


# Loading the data from GitHub

In [16]:
downloaded = []

for country in COUNTRIES:
    for group in GROUPS:
        filename = f"{country}_{group}.xlsx"
        url = f"{GITHUB_RAW_BASE}/{country}/{group}/{filename}"
        out_dir = RAW_ROOT / country / group
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / filename

        try:
            if not out_path.exists():
                print(f"Downloading: {url}")
                urlretrieve(url, out_path)
            else:
                print(f"Already exists: {out_path}")

            downloaded.append({
                "country": country,
                "group": group,
                "filename": filename,
                "path": out_path,
                "status": "ok",
            })
        except Exception as e:
            print(f"FAILED: {url} -> {e}")
            downloaded.append({
                "country": country,
                "group": group,
                "filename": filename,
                "path": out_path,
                "status": f"failed: {e}",
            })

download_df = pd.DataFrame(downloaded)
display(download_df)

Already exists: multilingual/data/raw/cultsportqa/China/HBQ/China_HBQ.xlsx
Already exists: multilingual/data/raw/cultsportqa/China/RBQ/China_RBQ.xlsx
Already exists: multilingual/data/raw/cultsportqa/China/SBQ/China_SBQ.xlsx
Already exists: multilingual/data/raw/cultsportqa/India/HBQ/India_HBQ.xlsx
Already exists: multilingual/data/raw/cultsportqa/India/RBQ/India_RBQ.xlsx
Already exists: multilingual/data/raw/cultsportqa/India/SBQ/India_SBQ.xlsx


,country,group,filename,path,status
0,China,HBQ,China_HBQ.xlsx,multilingual/data/raw/cultsportqa/China/HBQ/China_HBQ.xlsx,ok
1,China,RBQ,China_RBQ.xlsx,multilingual/data/raw/cultsportqa/China/RBQ/China_RBQ.xlsx,ok
2,China,SBQ,China_SBQ.xlsx,multilingual/data/raw/cultsportqa/China/SBQ/China_SBQ.xlsx,ok
3,India,HBQ,India_HBQ.xlsx,multilingual/data/raw/cultsportqa/India/HBQ/India_HBQ.xlsx,ok
4,India,RBQ,India_RBQ.xlsx,multilingual/data/raw/cultsportqa/India/RBQ/India_RBQ.xlsx,ok
5,India,SBQ,India_SBQ.xlsx,multilingual/data/raw/cultsportqa/India/SBQ/India_SBQ.xlsx,ok


In [17]:
workbook_rows = []

for row in downloaded:
    if row["status"] != "ok" or not row["path"].exists():
        continue

    try:
        xls = pd.ExcelFile(row["path"])
        for sheet in xls.sheet_names:
            df = pd.read_excel(row["path"], sheet_name=sheet, engine="openpyxl")
            workbook_rows.append({
                "country": row["country"],
                "group": row["group"],
                "file": row["filename"],
                "sheet": sheet,
                "n_rows": len(df),
                "columns": list(df.columns),
            })
    except Exception as e:
        workbook_rows.append({
            "country": row["country"],
            "group": row["group"],
            "file": row["filename"],
            "sheet": None,
            "n_rows": None,
            "columns": None,
            "error": str(e),
        })

workbook_df = pd.DataFrame(workbook_rows)
display(workbook_df)

,country,group,file,sheet,n_rows,columns
0,China,HBQ,China_HBQ.xlsx,cuju,151,"[question, option, answer]"
1,China,HBQ,China_HBQ.xlsx,dragon boat racing,129,"[question, option, answer]"
2,China,HBQ,China_HBQ.xlsx,go,124,"[question, option, answer]"
3,China,HBQ,China_HBQ.xlsx,jianzi,144,"[question, option, answer]"
4,China,HBQ,China_HBQ.xlsx,kung fu,133,"[question, option, answer]"
5,China,HBQ,China_HBQ.xlsx,tai chi,132,"[question, option, answer]"
6,China,RBQ,China_RBQ.xlsx,cuju,141,"[question, option, answer]"
7,China,RBQ,China_RBQ.xlsx,dragon boat racing,138,"[question, option, answer]"
8,China,RBQ,China_RBQ.xlsx,go,140,"[question, option, answer]"
9,China,RBQ,China_RBQ.xlsx,jianzi,134,"[question, option, answer]"


In [18]:
print("Files downloaded:", (download_df["status"] == "ok").sum(), "/", len(download_df))
print("Total sheets:", len(workbook_df))

display(
    workbook_df
    .groupby(["country", "group"], dropna=False)
    .agg(
        n_sheets=("sheet", "nunique"),
        total_rows=("n_rows", "sum"),
    )
    .reset_index()
    .sort_values(["country", "group"])
)

Files downloaded: 6 / 6
Total sheets: 63


,country,group,n_sheets,total_rows
0,China,HBQ,6,813
1,China,RBQ,6,841
2,China,SBQ,6,874
3,India,HBQ,15,6606
4,India,RBQ,15,6157
5,India,SBQ,15,6763


In [19]:
sports_by_country_group = (
    workbook_df
    .dropna(subset=["sheet"])
    .groupby(["country", "group"])["sheet"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="sports")
)

display(sports_by_country_group)

,country,group,sports
0,China,HBQ,"[cuju, dragon boat racing, go, jianzi, kung fu, tai chi]"
1,China,RBQ,"[cuju, dragon boat racing, go, jianzi, kung fu, tai chi]"
2,China,SBQ,"[cuju, dragon boat racing, go, jianzi, kung fu, tai chi]"
3,India,HBQ,"[Carrom, Chess, Dhop Khel, Gilli Danda, Kabaddi, Kalaripayattu, Kanche, Kho kho, Lagori, Malakhamb, Mukna, Musti Yuddha, Pambaram, Silambam, Thang Ta]"
4,India,RBQ,"[Carrom, Chess, Dhop Khel, Gilli Danda, Kabaddi, Kalaripayattu, Kanche, Kho kho, Lagori, Malakhamb, Mukna, Musti Yuddha, Pambaram, Silambam, Thang Ta]"
5,India,SBQ,"[Carrom, Chess, Dhop Khel, Gilli Danda, Kabaddi, Kalaripayattu, Kanche, Kho kho, Lagori, Malakhamb, Mukna, Musti Yuddha, Pambaram, Silambam, Thang Ta]"


In [20]:
for country in COUNTRIES:
    sub = sports_by_country_group[sports_by_country_group["country"] == country]
    if sub.empty:
        continue

    print(f"\n=== {country} ===")
    group_to_sports = dict(zip(sub["group"], sub["sports"]))

    all_sports = sorted(set().union(*group_to_sports.values())) if group_to_sports else []
    print("all sports:", all_sports)

    for group in GROUPS:
        sports = group_to_sports.get(group, [])
        missing = sorted(set(all_sports) - set(sports))
        extra = sorted(set(sports) - set(all_sports))
        print(f"{group}: {len(sports)} sports")
        if missing:
            print(f"  missing vs union: {missing}")
        if extra:
            print(f"  extra: {extra}")


=== China ===
all sports: ['cuju', 'dragon boat racing', 'go', 'jianzi', 'kung fu', 'tai chi']
HBQ: 6 sports
RBQ: 6 sports
SBQ: 6 sports

=== India ===
all sports: ['Carrom', 'Chess', 'Dhop Khel', 'Gilli Danda', 'Kabaddi', 'Kalaripayattu', 'Kanche', 'Kho kho', 'Lagori', 'Malakhamb', 'Mukna', 'Musti Yuddha', 'Pambaram', 'Silambam', 'Thang Ta']
HBQ: 15 sports
RBQ: 15 sports
SBQ: 15 sports


# Combined dataframe for all sheets
Load all sheets into a combined dataframe:

In [21]:
all_rows = []

for meta in workbook_rows:
    if meta.get("sheet") is None:
        continue

    path = RAW_ROOT / meta["country"] / meta["group"] / meta["file"]
    df = pd.read_excel(path, sheet_name=meta["sheet"], engine="openpyxl")

    # standardize column names lightly
    df.columns = [str(c).strip().lower() for c in df.columns]

    # keep raw columns if present
    for i, r in df.iterrows():
        all_rows.append({
            "country": meta["country"],
            "group": meta["group"],
            "sport": meta["sheet"],
            "row_idx": i,
            "question": r.get("question", np.nan),
            "option": r.get("option", np.nan),
            "answer": r.get("answer", np.nan),
        })

all_df = pd.DataFrame(all_rows)
print(all_df.shape)
display(all_df.head())

(22054, 7)


,country,group,sport,row_idx,question,option,answer
0,China,HBQ,cuju,0,蹴鞠这项中国古代运动的创始人是谁？,"['A) 黄帝', 'B) 秦昭襄王', 'C) 张郃', 'D) 孔子']",A) 黄帝
1,China,HBQ,cuju,1,蹴鞠在中国哪个朝代成为一项极为流行的运动？,"['A) 唐朝', 'B) 宋朝', 'C) 汉朝', 'D) 清朝']",B) 宋朝
2,China,HBQ,cuju,2,“蹴鞠”两个字的字面���思分别是什么？,"['A) 踢、球', 'B) 玩、游戏', 'C) 古代足球', 'D) 皇家运动']",A) 踢、球
3,China,HBQ,cuju,3,哪份历史文献最早记载了蹴鞠这项运动？,"['A)《史记》', 'B)《诗经》', 'C)《左传》', 'D)《春秋》']",A)《史记》
4,China,HBQ,cuju,4,唐朝哪位皇帝曾大力提拔蹴鞠？,A) 唐太宗 B) 玄宗 C) 武帝 D) 太祖,B) 唐玄宗


In [22]:
display(
    all_df
    .groupby(["country", "group"])
    .size()
    .reset_index(name="n_rows")
    .pivot(index="country", columns="group", values="n_rows")
    .fillna(0)
    .astype(int)
)

group,HBQ,RBQ,SBQ
country,,,
China,813,841,874
India,6606,6157,6763


In [23]:
display(
    all_df
    .groupby(["country", "group", "sport"])
    .size()
    .reset_index(name="n_rows")
    .sort_values(["country", "sport", "group"])
)

,country,group,sport,n_rows
0,China,HBQ,cuju,151
6,China,RBQ,cuju,141
12,China,SBQ,cuju,146
1,China,HBQ,dragon boat racing,129
7,China,RBQ,dragon boat racing,138
13,China,SBQ,dragon boat racing,146
2,China,HBQ,go,124
8,China,RBQ,go,140
14,China,SBQ,go,147
3,China,HBQ,jianzi,144


In [27]:
def show_samples(country=None, group=None, sport=None, n=5, seed=42, random=True):
    df = all_df.copy()

    if country is not None:
        df = df[df["country"] == country]
    if group is not None:
        df = df[df["group"] == group]
    if sport is not None:
        df = df[df["sport"] == sport]

    if df.empty:
        print("No rows match the filters.")
        return

    if random:
        out = df.sample(min(n, len(df)), random_state=seed)
    else:
        out = df.head(n)

    display(out[["country", "group", "sport", "row_idx", "question", "option", "answer"]])

In [28]:
for group in ["HBQ", "RBQ", "SBQ"]:
    print(f"\n=== India / Kabaddi / {group} ===")
    show_samples(country="India", group=group, sport="Kabaddi", n=5)


=== India / Kabaddi / HBQ ===


,country,group,sport,row_idx,question,option,answer
2880,India,HBQ,Kabaddi,352,कबड्डी में उस कौशल का नाम क्या है जिसमें डिफेंडर अपने शरीर का उपयोग करके रेडर के आगे बढ़ने का रास्ता रोकता है?,"['A) ब्लॉक', 'B) टैकल', 'C) होल्ड', 'D) पिन']",A) ब्लॉक
2567,India,HBQ,Kabaddi,39,भारत के किस राज्य में एक अनोखी कबड्डी शैली है जिसे 'राजस्थान कबड्डी' के नाम से जाना जाता है?,"ए) राजस्थान, बी) हरियाणा, सी) पंजाब, डी) उत्तर प्रदेश",A) राजस्थान
2889,India,HBQ,Kabaddi,361,प्रो कबड्डी लीग की फ्रेंचाइजी यू मुम्बा का घर कौन सा शहर है?,"ए) मुंबई, बी) दिल्ली, सी) बेंगलुरु, डी) चेन्नई",ए) मुंबई
2886,India,HBQ,Kabaddi,358,"कबड्डी के एक अंतरराष्ट्रीय प्रतियोगिता में, जिसमें सभी टीमें एक-दूसरे के साथ खेलती हैं और अंत में सबसे अधिक अंक वाली टीम जीतत�� है, उस प्रतियोगिता के लिए प्रयुक्त शब्द क्या है?","['A) नॉकआउट', 'B) राउंड-रॉबिन', 'C) लीग', 'D) उपरोक्त सभी']",B) राउंड-रॉबिन
2682,India,HBQ,Kabaddi,154,कौन सा कबड्डी खिलाड़ी अपने असाधारण रक्षात्मक कौशल के लिए जाना जाता है और 2019 कबड्डी विश्व कप में महत्वपूर्ण योगदानकर्ता था?,"['ए) मंजीत छिल्लर', 'बी) संदीप नरवाल', 'सी) नितिन तोमर', 'डी) अजय ठाकुर']",डी) अजय ठाक��र



=== India / Kabaddi / RBQ ===


,country,group,sport,row_idx,question,option,answer
9591,India,RBQ,Kabaddi,457,कबड्डी में रेडर को रेड पूरा करने के लिए दिए गए समय की अवधि के लिए क्या शब्द प्रयुक्त होता है?,"['ए) छापे का समय', 'बी) छापे की सीमा', 'सी) छापे की अवधि', 'डी) छापे की घड़ी']",सी) छापे की अवधि
9207,India,RBQ,Kabaddi,73,'टाइमआउट' टीमों को क्या करने की अनुमति देता है?,"A) रणनीतियों पर चर्चा करें और आराम करें, B) खेल की अवधि बदलें, C) टीम लाइनअप बदलें, D) अंक प्रदान करें",A) रणनीतियों पर चर्चा करें और आराम करें
9365,India,RBQ,Kabaddi,231,कबड्डी में लाल कार्ड जारी करने का स्वीकार्य कारण क्या है?,"['ए) गंभीर कदाचार', 'बी) तकनीकी गड़बड़ी', 'सी) बार-बार छोटे उल्लंघन', 'डी) खेल में देरी']",ए) गंभीर कदाचार
9309,India,RBQ,Kabaddi,175,कबड्डी में 'सुपर टैकल' की क्या परिभाषा है?,"['A) तीन या अधिक डिफेंडरों द्वारा सफल टैकल', 'B) कई टच के साथ सफल रेड', 'C) दो डिफेंडरों द्वारा टैकल', 'D) बोनस अंक अर्जित करने वाली रेड']",A) तीन या अधिक डिफेंडरों द्वारा सफल टैकल
9371,India,RBQ,Kabaddi,237,कबड्डी में सफल 'सुपर टैकल' के लिए कितने अंक दिए जाते हैं?,"['A) 2 अंक', 'B) 3 अंक', 'C) 5 अंक', 'D) 1 अंक']",B) 3 अंक



=== India / Kabaddi / SBQ ===


,country,group,sport,row_idx,question,option,answer
15812,India,SBQ,Kabaddi,521,"कबड्डी रेड के दौरान यदि रेडर पकड़ा नहीं जाता है और बचाव करने वाली टीम के पास 3 या कम खिलाड़ी होते हैं, तो क्या होता है?","['A) बचाव करने वाली टीम को 1 अंक मिलता है', 'B) रेडर आउट हो जाता है', 'C) रेड करने वाली टीम को 1 अंक मिलता है', 'D) खेल रोक दिया जाता है']",C) रेड करने वाली टीम को 1 अंक मिलता है
15679,India,SBQ,Kabaddi,388,"कबड्डी में, यदि कोई रेडर विपक्षी टीम के कोर्ट में प्रवेश करता है और डिफेंडर उसे तुरंत पकड़ लेते हैं, तो क्या परिणाम होगा?","['A) रेडर को 1 अंक मिलता है', 'B) र���डर आउट हो जाता है', 'C) बचाव करने वाली टीम को 1 अंक मिलता है', 'D) खेल रोक दिया जाता है']",B) रेडर आउट हो जाता है
15297,India,SBQ,Kabaddi,6,निम्नलिखित में से कौन सी कबड्डी में वैध रक्षात्मक तकनीक नहीं है?,"['A) एंकल होल्ड', 'B) ब्लॉक', 'C) चेन टैकल', 'D) गले का दबाव']",D) गले का दबाव
15475,India,SBQ,Kabaddi,184,"यदि कोई रेडर दो डिफेंडरों को छूता है और उसे एक डिफेंडर द्वारा टैकल किया जाता है जो सीमा से बाहर है, तो परिणाम क्या होगा?","['A) रेडर को 2 अंक मिलते हैं और टैकल अमान्य है', 'B) रेडर आउट हो जाता है', 'C) बचाव करने वाली टीम को 1 अंक मिलता है', 'D) रेड रद्द हो जाती है']",A) रेडर को 2 अंक मिलते हैं और टैकल अमान्य है
15369,India,SBQ,Kabaddi,78,एक रेडर ने डिफेंडर को छुआ और डिफेंडर ने दावा किया कि रेडर ने अवैध रूप से छुआ। अगर कोई फ़ाउल नहीं पाया जाता है तो फ़ैसला क्या होगा?,"['A) डिफेंडर आउट हो गया है', 'B) रेडर आउट हो गया है', 'C) रेड सफल है', 'D) कोई अंक नहीं दिया गया है']",C) रेड सफल है


In [29]:
for group in ["HBQ", "RBQ", "SBQ"]:
    print(f"\n=== China / kung fu / {group} ===")
    show_samples(country="China", group=group, sport="kung fu", n=5)


=== China / kung fu / HBQ ===


,country,group,sport,row_idx,question,option,answer
599,China,HBQ,kung fu,51,少林功夫被认为是哪个佛教宗派创立的？,A) 禅 B) 净土 C) 天台 D) 华严,A) 禅宗
617,China,HBQ,kung fu,69,在哪个历史时期，白莲教起义者广泛使用中国武术？,"['A) 明朝', 'B) 清朝', 'C) 唐朝', 'D) 汉朝']",B) 清朝
579,China,HBQ,kung fu,31,哪位历史人物被广泛认为是现代功夫之父？,A) 李小龙、B) 黄飞鸿、C) 张三丰、D) 陈望庭,A) 李小龙
590,China,HBQ,kung fu,42,在传统功夫训练体系中，“六和”至关重要。下列哪一项代表了六和之一？,"['A) 身心协调', 'B) 力量与敏捷的和谐', 'C) 心与意的和谐', 'D) 手与武器���和谐']",C) 心与意的和谐
665,China,HBQ,kung fu,117,功夫练习中，'气功'主要用来培养：,"['A) 内能', 'B) 力量', 'C) 速度', 'D) 杂技']",A) 内能



=== China / kung fu / RBQ ===


,country,group,sport,row_idx,question,option,answer
1474,China,RBQ,kung fu,108,什么是“气功”练习？,"['A) 能量修炼', 'B) 高踢', 'C) 拳击组合', 'D) 武器训练']",A) 能量修炼
1433,China,RBQ,kung fu,67,在功夫哲学中，“阴”和“阳”的概念指的是哪两种元素之间的平衡？,"['A) 光明与黑暗', 'B) 坚硬与柔软', 'C) 生与死', 'D) 速度与力量']",B) 坚硬与柔软
1397,China,RBQ,kung fu,31,功夫中的“气功”一词主要指的是：,"['A）呼吸与能量调节练习', 'B）高踢腿', 'C）肘击', 'D）实战练习']",A）呼吸与能量调节练习
1485,China,RBQ,kung fu,119,哪种踢法可以用来出其不意地攻击对手？,A) 侧踢 B) 前踢 C) 回旋踢 D) 低踢,C) 回旋踢
1408,China,RBQ,kung fu,42,在五禽戏中，哪种动物代表敏捷和智慧？,"['A) 猴子', 'B) 老虎', 'C) 鹿', 'D) 熊']",A) 猴子



=== China / kung fu / SBQ ===


,country,group,sport,row_idx,question,option,answer
2159,China,SBQ,kung fu,69,对手从意想不到的角度发起攻击，你应该如何应对？,"['A) 调整防御姿势并反击', 'B) 无视', 'C) 向前迈一步', 'D) 后退']",A) 调整防御姿势并反击
2230,China,SBQ,kung fu,140,对手抓住你的手腕。最佳应对措施是什么？,"['A) 解除控制', 'B) 无视', 'C) 后退', 'D) 让步']",A) 解除控制
2117,China,SBQ,kung fu,27,一位功夫高手使用了一种名为“铁砂掌”的技巧。这种技巧的目的是什么？,A) 提高速度，B) 提高精准度，C) 增强掌力，D) 增强柔韧性,C) 增强掌力，以便更有力地击打
2109,China,SBQ,kung fu,19,在一场功夫表演中，一名练习者使出一记跳跃旋踢。这一精彩动作的主要目的是什么？,"['A) 为了展现敏捷性', 'B) 为了出其不意地攻击对手', 'C) 为了增强攻击力', 'D) 为了躲避']",A) 为了展现敏捷性
2132,China,SBQ,kung fu,42,在一场武术比赛中，一名选手通过翻滚来躲避攻击。在这种情况下，翻滚的目的是什么？,"['A) 避免被击中并保持移动能力', 'B) 消耗对手体力', 'C) 快速重新定位', 'D) 增强力量']",A) 避免被��中并保持移动能力


# Convert Excel files to raw CSV files

In [40]:
# cleaning the files: convert each Excel sheet to CSV + save one combined raw CSV
def safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r"[^\w\s\-]", "", name, flags=re.UNICODE)
    name = re.sub(r"\s+", "_", name)
    return name


all_raw_rows = []

for xlsx_path in sorted(RAW_ROOT.glob("*/*/*.xlsx")):
    country = xlsx_path.parts[-3]
    group = xlsx_path.parts[-2]

    print(f"Processing {country}/{group}: {xlsx_path.name}")

    xls = pd.ExcelFile(xlsx_path, engine="openpyxl")

    for sheet in xls.sheet_names:
        df = pd.read_excel(xlsx_path, sheet_name=sheet, engine="openpyxl")

        # standardize column names minimally
        df.columns = [str(c).strip().lower() for c in df.columns]

        expected_cols = ["question", "option", "answer"]
        missing = [c for c in expected_cols if c not in df.columns]
        if missing:
            print(f"  [skip] {sheet}: missing columns {missing}")
            continue

        df = df[expected_cols].copy()

        # add metadata columns
        df.insert(0, "country", country)
        df.insert(1, "group", group)
        df.insert(2, "sport", sheet)
        df.insert(3, "row_idx", range(len(df)))

        # save one CSV per sheet
        out_dir = RAW_CSV_ROOT / country / group
        out_dir.mkdir(parents=True, exist_ok=True)

        out_path = out_dir / f"{safe_filename(sheet)}.csv"
        df.to_csv(out_path, index=False, encoding="utf-8")

        print(f"  saved {sheet}: {len(df)} rows -> {out_path}")

        all_raw_rows.append(df)

Processing China/HBQ: China_HBQ.xlsx
  saved cuju: 151 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cultsportqa_csv/China/HBQ/cuju.csv
  saved dragon boat racing: 129 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cultsportqa_csv/China/HBQ/dragon_boat_racing.csv
  saved go: 124 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cultsportqa_csv/China/HBQ/go.csv
  saved jianzi: 144 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cultsportqa_csv/China/HBQ/jianzi.csv
  saved kung fu: 133 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cultsportqa_csv/China/HBQ/kung_fu.csv
  saved tai chi: 132 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cultsportqa_csv/China/HBQ/tai_chi.csv
Processing China/RBQ: China_RBQ.xlsx
  saved cuju: 141 rows -> /s

In [41]:
# save one combined raw CSV
if all_raw_rows:
    raw_all_df = pd.concat(all_raw_rows, ignore_index=True)
    combined_path = RAW_CSV_ROOT / "cultsportqa_raw_all.csv"
    raw_all_df.to_csv(combined_path, index=False, encoding="utf-8")

    print(f"\nSaved combined raw CSV: {combined_path}")
    print(f"Total rows: {len(raw_all_df)}")

    display(raw_all_df.head())

    display(
        raw_all_df
        .groupby(["country", "group", "sport"])
        .size()
        .reset_index(name="n_rows")
        .sort_values(["country", "sport", "group"])
    )
else:
    raw_all_df = pd.DataFrame()
    print("No rows saved.")


Saved combined raw CSV: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/cultsportqa_csv/cultsportqa_raw_all.csv
Total rows: 22054


,country,group,sport,row_idx,question,option,answer
0,China,HBQ,cuju,0,蹴鞠这项中国古代运动的创始人是谁？,"['A) 黄帝', 'B) 秦昭襄王', 'C) 张郃', 'D) 孔子']",A) 黄帝
1,China,HBQ,cuju,1,蹴鞠在中国哪个朝代成为一项极为流行的运动？,"['A) 唐朝', 'B) 宋朝', 'C) 汉朝', 'D) 清朝']",B) 宋朝
2,China,HBQ,cuju,2,“蹴鞠”两个字的字面���思分别是什么？,"['A) 踢、球', 'B) 玩、游戏', 'C) 古代足球', 'D) 皇家运动']",A) 踢、球
3,China,HBQ,cuju,3,哪份历史文献最早记载了蹴鞠这项运动？,"['A)《史记》', 'B)《诗经》', 'C)《左传》', 'D)《春秋》']",A)《史记》
4,China,HBQ,cuju,4,唐朝哪位皇帝曾大力提拔蹴鞠？,A) 唐太宗 B) 玄宗 C) 武帝 D) 太祖,B) 唐玄宗


,country,group,sport,n_rows
0,China,HBQ,cuju,151
6,China,RBQ,cuju,141
12,China,SBQ,cuju,146
1,China,HBQ,dragon boat racing,129
7,China,RBQ,dragon boat racing,138
13,China,SBQ,dragon boat racing,146
2,China,HBQ,go,124
8,China,RBQ,go,140
14,China,SBQ,go,147
3,China,HBQ,jianzi,144


# Parse, filter, deduplicate, and save cleaned CultSportQA

In [42]:
raw_path = RAW_CSV_ROOT / "cultsportqa_raw_all.csv"
raw_df = pd.read_csv(raw_path)

print(f"Loaded raw rows: {len(raw_df)}")
display(raw_df.head())

Loaded raw rows: 22054


,country,group,sport,row_idx,question,option,answer
0,China,HBQ,cuju,0,蹴鞠这项中国古代运动的创始人是谁？,"['A) 黄帝', 'B) 秦昭襄王', 'C) 张郃', 'D) 孔子']",A) 黄帝
1,China,HBQ,cuju,1,蹴鞠在中国哪个朝代成为一项极为流行的运动？,"['A) 唐朝', 'B) 宋朝', 'C) 汉朝', 'D) 清朝']",B) 宋朝
2,China,HBQ,cuju,2,“蹴鞠”两个字的字面���思分别是什么？,"['A) 踢、球', 'B) 玩、游戏', 'C) 古代足球', 'D) 皇家运动']",A) 踢、球
3,China,HBQ,cuju,3,哪份历史文献最早记载了蹴鞠这项运动？,"['A)《史记》', 'B)《诗经》', 'C)《左传》', 'D)《春秋》']",A)《史记》
4,China,HBQ,cuju,4,唐朝哪位皇帝曾大力提拔蹴鞠？,A) 唐太宗 B) 玄宗 C) 武帝 D) 太祖,B) 唐玄宗


## Helpers

In [43]:
# Parsing / cleaning helpers

LETTER_MAP = {
    # Latin labels
    "A": "A", "B": "B", "C": "C", "D": "D",
    "a": "A", "b": "B", "c": "C", "d": "D",

    # Hindi/Devanagari labels observed in the files
    "ए": "A", "बी": "B", "सी": "C", "डी": "D",
    "अ": "A", "ब": "B", "स": "C", "द": "D",
}

ANSWER_LABEL_RE = re.compile(
    r"^\s*(A|B|C|D|a|b|c|d|ए|बी|सी|डी|अ|ब|स|द)\s*[\)\]）\.:\-]?",
    re.UNICODE,
)

OPTION_LABEL_RE = re.compile(
    r"(A|B|C|D|a|b|c|d|ए|बी|सी|डी|अ|ब|स|द)\s*[\)\]）\.:\-]",
    re.UNICODE,
)

COUNTRY_TO_LANGUAGE = {
    "China": "chinese",
    "India": "hindi",
}

GROUP_DESCRIPTION = {
    "HBQ": "historical_background",
    "RBQ": "rules_reference",
    "SBQ": "scenario_based",
}


def normalize_text(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_for_dedup(x) -> str:
    x = normalize_text(x).lower()
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def is_placeholder(x) -> bool:
    s = normalize_text(x).lower()
    return s in {
        "",
        "nan",
        "none",
        "error",
        "no correction needed",
        "no answer needed",
    }


def has_corruption(*fields) -> bool:
    return any("�" in normalize_text(x) for x in fields)


def parse_answer_label(answer):
    s = normalize_text(answer)
    if is_placeholder(s):
        return None

    m = ANSWER_LABEL_RE.match(s)
    if not m:
        return None

    raw_label = m.group(1)
    return LETTER_MAP.get(raw_label)


def parse_options(option_text):
    s = normalize_text(option_text)
    if is_placeholder(s):
        return None

    # Case 1: Python-list-looking option field:
    # "['A) ...', 'B) ...', 'C) ...', 'D) ...']"
    if s.startswith("[") and s.endswith("]"):
        try:
            parts = ast.literal_eval(s)
        except Exception:
            return None

        if not isinstance(parts, list):
            return None

        parts = [normalize_text(p) for p in parts]

    # Case 2: inline option field:
    # "A) ... B) ... C) ... D) ..."
    else:
        matches = list(OPTION_LABEL_RE.finditer(s))
        if not matches:
            return None

        parts = []
        for i, match in enumerate(matches):
            start = match.start()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(s)
            parts.append(s[start:end].strip())

    parsed = {}

    for part in parts:
        m = ANSWER_LABEL_RE.match(part)
        if not m:
            return None

        raw_label = m.group(1)
        label = LETTER_MAP.get(raw_label)
        if label is None:
            return None

        text = part[m.end():].strip()
        text = re.sub(r"^[\)\]）\.:\-\s、,，]+", "", text).strip()
        text = text.rstrip("、,，;；").strip()

        if not text:
            return None
        if label in parsed:
            return None

        parsed[label] = text

    if set(parsed.keys()) != {"A", "B", "C", "D"}:
        return None

    return parsed

## Clean and rejected dataframes

In [44]:
# Build cleaned and rejected dataframes

clean_rows = []
rejected_rows = []

for _, row in raw_df.iterrows():
    country = normalize_text(row.get("country"))
    group = normalize_text(row.get("group"))
    sport = normalize_text(row.get("sport"))
    row_idx = row.get("row_idx")

    question = normalize_text(row.get("question"))
    option_raw = normalize_text(row.get("option"))
    answer_raw = normalize_text(row.get("answer"))

    reject_reason = None

    if country not in COUNTRY_TO_LANGUAGE:
        reject_reason = "unsupported_country"
    elif is_placeholder(question):
        reject_reason = "invalid_question"
    elif is_placeholder(option_raw):
        reject_reason = "invalid_options"
    elif is_placeholder(answer_raw):
        reject_reason = "invalid_answer"
    elif has_corruption(question, option_raw, answer_raw):
        reject_reason = "corruption"
    else:
        options = parse_options(option_raw)
        answer = parse_answer_label(answer_raw)

        if options is None:
            reject_reason = "unparseable_options"
        elif answer is None:
            reject_reason = "unparseable_answer"
        else:
            prompt = (
                f"{question}\n\n"
                f"A) {options['A']}\n"
                f"B) {options['B']}\n"
                f"C) {options['C']}\n"
                f"D) {options['D']}"
            )

            clean_rows.append({
                "dataset": "cultsportqa",
                "country": country,
                "language": COUNTRY_TO_LANGUAGE[country],
                "group": group,
                "group_description": GROUP_DESCRIPTION.get(group, group),
                "sport": sport,
                "row_idx": int(row_idx) if pd.notna(row_idx) else None,
                "question": question,
                "option_A": options["A"],
                "option_B": options["B"],
                "option_C": options["C"],
                "option_D": options["D"],
                "prompt": prompt,
                "answer": answer,
                "raw_option": option_raw,
                "raw_answer": answer_raw,
            })

    if reject_reason is not None:
        rejected_rows.append({
            "country": country,
            "group": group,
            "sport": sport,
            "row_idx": int(row_idx) if pd.notna(row_idx) else None,
            "question": question,
            "option": option_raw,
            "answer": answer_raw,
            "reject_reason": reject_reason,
        })

clean_df = pd.DataFrame(clean_rows)
rejected_df = pd.DataFrame(rejected_rows)

print(f"Raw rows:      {len(raw_df)}")
print(f"Clean rows:    {len(clean_df)}")
print(f"Rejected rows: {len(rejected_df)}")

display(rejected_df["reject_reason"].value_counts())
display(clean_df.head())

Raw rows:      22054
Clean rows:    15259
Rejected rows: 6795


reject_reason
corruption             6185
unparseable_options     450
invalid_question         71
unparseable_answer       60
invalid_answer           29
Name: count, dtype: int64

,dataset,country,language,group,group_description,sport,row_idx,question,option_A,option_B,option_C,option_D,prompt,answer,raw_option,raw_answer
0,cultsportqa,China,chinese,HBQ,historical_background,cuju,0,蹴鞠这项中国古代运动的创始人是谁？,黄帝,秦昭襄王,张郃,孔子,蹴鞠这项中国古代运动的创始人是谁？\n\nA) 黄帝\nB) 秦昭襄王\nC) 张郃\nD) 孔子,A,"['A) 黄帝', 'B) 秦昭襄王', 'C) 张郃', 'D) 孔子']",A) 黄帝
1,cultsportqa,China,chinese,HBQ,historical_background,cuju,1,蹴鞠在中国哪个朝代成为一项极为流行的运动？,唐朝,宋朝,汉朝,清朝,蹴鞠在中国哪个朝代成为一项极为流行的运动？\n\nA) 唐朝\nB) 宋朝\nC) 汉朝\nD) 清朝,B,"['A) 唐朝', 'B) 宋朝', 'C) 汉朝', 'D) 清朝']",B) 宋朝
2,cultsportqa,China,chinese,HBQ,historical_background,cuju,3,哪份历史文献最早记载了蹴鞠这项运动？,《史记》,《诗经》,《左传》,《春秋》,哪份历史文献最早记载了蹴鞠这项运动？\n\nA) 《史记》\nB) 《诗经》\nC) 《左传》\nD) 《春秋》,A,"['A)《史记》', 'B)《诗经》', 'C)《左传》', 'D)《春秋》']",A)《史记》
3,cultsportqa,China,chinese,HBQ,historical_background,cuju,4,唐朝哪位皇帝曾大力提拔蹴鞠？,唐太宗,玄宗,武帝,太祖,唐朝哪位皇帝曾大力提拔蹴鞠？\n\nA) 唐太宗\nB) 玄宗\nC) 武帝\nD) 太祖,B,A) 唐太宗 B) 玄宗 C) 武帝 D) 太祖,B) 唐玄宗
4,cultsportqa,China,chinese,HBQ,historical_background,cuju,5,蹴鞠这项运动起源于哪个古代文明？,中国,印度,希腊,罗马,蹴鞠这项运动起源于哪个古代文明？\n\nA) 中国\nB) 印度\nC) 希腊\nD) 罗马,A,A) 中国，B) 印度，C) 希腊，D) 罗马,A) 中国


## Dedup clean rows

In [45]:
# Deduplicate cleaned rows

clean_df["dedup_key"] = (
    clean_df["country"].map(normalize_for_dedup)
    + "\n" + clean_df["language"].map(normalize_for_dedup)
    + "\n" + clean_df["prompt"].map(normalize_for_dedup)
    + "\n" + clean_df["answer"].map(normalize_for_dedup)
)

n_before = len(clean_df)
clean_df = clean_df.drop_duplicates("dedup_key").copy()
n_after = len(clean_df)

print(f"Before dedup: {n_before}")
print(f"After dedup:  {n_after}")
print(f"Removed:      {n_before - n_after}")

clean_df = clean_df.drop(columns=["dedup_key"])

Before dedup: 15259
After dedup:  15239
Removed:      20


In [ ]:
# inspect cleaned counts, per category in HBQ, RBQ, SBQ
display(
    clean_df
    .groupby(["country", "group"])
    .size()
    .reset_index(name="n_clean")
    .pivot(index="country", columns="group", values="n_clean")
    .fillna(0)
    .astype(int)
)

group,HBQ,RBQ,SBQ
country,,,
China,791,814,824
India,4914,4003,3893


In [57]:
display(
    clean_df
    .groupby(["country", "language"])
    .size()
    .reset_index(name="n_clean")
)

,country,language,n_clean
0,China,chinese,2429
1,India,hindi,12810


In [56]:
display(
    clean_df
    .groupby(["country", "group", "sport"])
    .size()
    .reset_index(name="n_clean")
    .sort_values(["country", "sport", "group"])
)

,country,group,sport,n_clean
0,China,HBQ,cuju,147
6,China,RBQ,cuju,139
12,China,SBQ,cuju,142
1,China,HBQ,dragon boat racing,126
7,China,RBQ,dragon boat racing,135
13,China,SBQ,dragon boat racing,132
2,China,HBQ,go,122
8,China,RBQ,go,134
14,China,SBQ,go,133
3,China,HBQ,jianzi,143


In [ ]:
# inspect random cleaned examples
display(
    clean_df
    .sample(min(20, len(clean_df)), random_state=42)
    [["country", "language", "group", "sport", "prompt", "answer"]]
)

,country,language,group,sport,prompt,answer
2621,India,hindi,HBQ,Kabaddi,2019 प्रो कबड्डी लीग सीज़न में किस कबड्डी खिलाड़ी ने सबसे अधिक सफल टैकल हासिल किए?\n\nA) संदीप नरवाल\nB) मोहित छिल्लर\nC) नितिन तोमर\nD) अजय ठाकुर,B
14953,India,hindi,SBQ,Kalaripayattu,"कलारीपयट्टू में, कौन सा अभ्यास लचीलेपन और समग्र शरीर की कंडीशनिंग को बढ़ाने पर केंद्रित है?\n\nA) उझिचिल\nB) एडवस\nC) मार्मा\nD) चुवाडु",A
14981,India,hindi,SBQ,Kalaripayattu,"कलारीपयट्टू में, कौन सा अभ्यास प्रभावी बचाव और प्रतिकार के लिए फुटवर्क को बढ़ाने पर केंद्रित है?\n\nA) चुवाडु\nB) वर्मा\nC) मार्मा\nD) एडवस",A
14721,India,hindi,SBQ,Mukna,"खिलाड़ी X और खिलाड़ी Y एक मुक्केबाजी मुकाबले में प्रतिस्पर्धा कर रहे हैं। अगर दोनों खिलाड़ी 2 राउंड जीतते हैं और मैच बराबरी पर है, तो आगे क्या होगा?\n\nA) मैच तब तक जारी रहता है जब तक कि एक खिलाड़ी दूसरा राउंड नहीं जीत लेता।\nB) अंतिम निर्णायक राउंड खेला जाता है।\nC) मैच को ड्रॉ घोषित कर दिया जाता है।\nD) सबसे कम फॉल वाले खिलाड़ी को विजेता घोषित किया जाता है।",B
487,China,chinese,HBQ,jianzi,哪个朝代提倡在节庆期间进行蹴鞠活动？\n\nA) 汉\nB) 唐\nC) 明\nD) 清,B
9639,India,hindi,RBQ,Carrom,यदि कोई खिलाड़ी एक ही शॉट में क्वीन और स्ट्राइकर दोनों को पॉकेट में डाल दे तो क्या होगा?\n\nA) खिलाड़ी अपनी बारी खो देता है\nB) दोनों वापस कर दिए जाते हैं\nC) रानी वापस कर दी जाती है\nD) कैरम वापस कर दिए जाते हैं,A
12024,India,hindi,SBQ,Kho kho,"एक नाज़ुक मौके पर, टीम B का चेज़र एक डिफेंडर को टैग करता है जो बाउंड्री लाइन के अंदर दौड़ रहा था और टैग के समय उसका एक पैर बाउंड्री लाइन पर था। सही नियम क्या है?\n\nA) डिफेंडर बाहर है\nB) डिफेंडर सुरक्षित है\nC) राउंड पुनः शुरू होता है\nD) दोनों चेज़र बाहर हैं",B
2465,India,hindi,HBQ,Kabaddi,किस भारतीय राज्य में कबड्डी का एक प्रकार 'कर्नाटक कबड्डी' के नाम से जाना जाता है?\n\nA) कर्नाटक\nB) केरल\nC) आंध्र प्रदेश\nD) तेलंगाना,A
2331,China,chinese,SBQ,tai chi,练习太极拳时，为什么保持放松自然的姿势很重要？\n\nA) 提高速度和攻击性\nB) 避免肌肉拉伤并保持平衡\nC) 专注于外部技巧\nD) 产生高强度运动,B
339,China,chinese,HBQ,go,哪种围棋等级系统适合从初学者进阶到高级棋手的玩家？\n\nA) 级\nB) 段\nC) 初级\nD) 老师,B


## Save clean, rejected and per-language files

In [ ]:
# save cleaned and rejected files
clean_out = PROCESSED_ROOT / "cultsportqa_clean_all.csv"
rejected_out = PROCESSED_ROOT / "cultsportqa_rejected_all.csv"

clean_df.to_csv(clean_out, index=False, encoding="utf-8")
rejected_df.to_csv(rejected_out, index=False, encoding="utf-8")

print(f"Saved clean data:    {clean_out}")
print(f"Saved rejected data: {rejected_out}")

Saved clean data:    /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cultsportqa/cultsportqa_clean_all.csv
Saved rejected data: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cultsportqa/cultsportqa_rejected_all.csv


In [54]:
# save per language files
for language, lang_df in df.groupby("language"):
    out_path = PROCESSED_ROOT / f"cultsportqa_clean_{language}.csv"
    lang_df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved {len(lang_df)} rows -> {out_path}")

Saved 2429 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cultsportqa/cultsportqa_clean_chinese.csv
Saved 12810 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/cultsportqa/cultsportqa_clean_hindi.csv
